# Semantic Chunking — A Deep Dive

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/semantic_chunking.ipynb)

> **Goal:** Instead of splitting text at fixed character counts, split where *meaning shifts*.
> We detect natural breakpoints by embedding every sentence and watching for drops in cosine similarity between neighbours.

**What you will learn:**
1. Why fixed-size chunking loses context at boundaries.
2. The full semantic-chunking algorithm, implemented from scratch.
3. How to visualize the similarity curve and pick thresholds (percentile, absolute, std-dev).
4. A sliding-window variant that smooths noisy signals.
5. A complete RAG pipeline: semantic chunks → FAISS → retrieve → generate with **Qwen3-8B**.
6. Head-to-head comparison of fixed-size vs. semantic chunking.
7. Hierarchical semantic chunking — recursive splitting at paragraph then sentence level.

**Stack:** `sentence-transformers (bge-base-en-v1.5)`, `FAISS`, `transformers + bitsandbytes (Qwen3-8B 4-bit)`, `scikit-learn`, `numpy`, `matplotlib`.
No LangChain, no LlamaIndex, no OpenAI API — everything runs locally.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **What Is Semantic Chunking?**
- Understand **The Algorithm at a Glance**
- Understand **Synthetic Corpus**
- Understand **Implementing from Scratch**
- Understand **Visualizing the Similarity Curve**


## 1 — What Is Semantic Chunking?

Traditional chunking slices a document every *N* characters (or tokens). That is fast but blind:

| Fixed-Size (300 chars) | Semantic Chunking |
|---|---|
| Cuts mid-sentence or mid-paragraph | Cuts **between** topics |
| Adjacent chunks may duplicate or orphan a thought | Each chunk is a self-contained semantic unit |
| Chunk boundaries are arbitrary | Boundaries align with meaning shifts |

**Core idea:** Embed every sentence, compute the cosine similarity between consecutive
sentence embeddings, and declare a *breakpoint* wherever that similarity drops
significantly. Sentences between two breakpoints form one chunk.

This was first popularized by [Greg Kamradt](https://youtu.be/8OJC21T2SL4?t=1933) and
later integrated into libraries like LangChain's `SemanticChunker`. Here we build it
from scratch so you understand every moving part.

## 2 — The Algorithm at a Glance

```
Input : A long document (string)
Output: A list of semantically coherent chunks

1. SPLIT the document into sentences  [s₁, s₂, …, sₙ]
2. EMBED each sentence               [e₁, e₂, …, eₙ]
3. COMPUTE cosine similarity between consecutive pairs:
       sim_i = cos(eᵢ, eᵢ₊₁)    for i = 1 … n-1
4. IDENTIFY breakpoints where sim_i drops below a threshold T
       breakpoints = { i : sim_i < T }
5. GROUP sentences between consecutive breakpoints into chunks
```

The only design choices are:
- **Which embedding model** to use (we pick `bge-base-en-v1.5`, a strong open-source model).
- **How to set T** — absolute value, percentile of the similarity distribution, or standard-deviation based.
- Whether to **smooth** the similarity signal first (sliding window).

In [ ]:
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy scikit-learn matplotlib

In [ ]:
import os, pathlib

# Google Drive cache for large models
CACHE = "/content/drive/MyDrive/models"
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    CACHE = str(pathlib.Path.home() / ".cache" / "huggingface" / "hub")

os.makedirs(CACHE, exist_ok=True)
os.environ["HF_HOME"]           = CACHE
os.environ["TRANSFORMERS_CACHE"] = CACHE
os.environ["HF_HUB_CACHE"]      = CACHE
print(f"Model cache → {CACHE}")

In [ ]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

# ── Embedding model ──────────────────────────────────────────
EMB_ID = "BAAI/bge-base-en-v1.5"
embedder = SentenceTransformer(EMB_ID, cache_folder=CACHE)
print(f"Embedder loaded: {EMB_ID}  dim={embedder.get_sentence_embedding_dimension()}")

# ── LLM: Qwen3-8B  4-bit NF4 ───────────────────────────────
LLM_ID = "Qwen/Qwen3-8B"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(LLM_ID, cache_dir=CACHE)
model = AutoModelForCausalLM.from_pretrained(
    LLM_ID, quantization_config=bnb, device_map="auto", cache_dir=CACHE,
)
print(f"LLM loaded: {LLM_ID}  (4-bit NF4)")

In [ ]:
def generate(prompt, max_new_tokens=512, temperature=0.7):
    """Generate a response from Qwen3-8B with /no_think for deterministic output."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": prompt + "\n/no_think"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, top_p=0.9, do_sample=True,
        )
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

# Quick smoke test
print(generate("Say hello in one sentence.", max_new_tokens=30))

## 3 — Synthetic Corpus

We craft **18 paragraphs** spanning **4 distinct topics** so that semantic boundaries
are clearly visible when we plot the similarity curve.

| Paragraphs | Topic |
|---|---|
| 1–5 | **Photosynthesis & Plant Biology** |
| 6–9 | **The History of the Internet** |
| 10–14 | **Basics of Cooking & Nutrition** |
| 15–18 | **Space Exploration & Mars Missions** |

Each topic shift should produce a visible dip in the cosine-similarity signal.

In [ ]:
CORPUS = """
Photosynthesis is the process by which green plants convert sunlight into chemical energy. Chloroplasts in the leaf cells contain chlorophyll, a pigment that absorbs light primarily in the blue and red wavelengths. The light-dependent reactions take place in the thylakoid membranes, where water molecules are split to release oxygen.

The Calvin cycle, also called the light-independent reactions, occurs in the stroma of the chloroplast. Carbon dioxide from the atmosphere is fixed into organic molecules through a series of enzyme-catalysed steps. The key enzyme RuBisCO catalyses the attachment of CO₂ to ribulose bisphosphate.

Plants have evolved several adaptations to maximise photosynthetic efficiency. C4 plants, such as maize and sugarcane, concentrate CO₂ around RuBisCO to reduce photorespiration. CAM plants, like cacti, open their stomata at night to minimise water loss in arid environments.

Root systems play an essential role in supporting photosynthesis by absorbing water and minerals from the soil. Mycorrhizal fungi form symbiotic relationships with plant roots, greatly extending the effective surface area for nutrient absorption. Nitrogen fixation by rhizobial bacteria in legume root nodules provides a natural source of fertiliser.

Transpiration drives the movement of water from roots to leaves through the xylem. The cohesion-tension theory explains how hydrogen bonds between water molecules create a continuous column under negative pressure. Guard cells surrounding each stoma regulate gas exchange and water loss.

The history of the Internet begins in the late 1960s with ARPANET, a project funded by the United States Department of Defense. The network initially connected four universities and used packet switching, a revolutionary method for transmitting data in small bundles rather than continuous streams.

By the 1980s, the adoption of TCP/IP as the standard communication protocol unified disparate networks into a single global system. The Domain Name System was introduced in 1984, allowing users to access servers by human-readable names instead of numeric IP addresses.

Tim Berners-Lee invented the World Wide Web in 1989 at CERN, introducing HTML, URLs, and the first web browser. The release of the Mosaic browser in 1993 brought graphical browsing to the masses and sparked the dot-com boom of the late 1990s.

The modern Internet supports billions of connected devices through technologies such as fibre optics, 5G wireless, and satellite links. Cloud computing has shifted infrastructure from local servers to massive data centres, enabling services like streaming, social media, and real-time collaboration on a global scale.

Cooking is both a practical skill and an art form that has been central to human culture for millennia. Applying heat transforms the chemical structure of food: proteins denature, starches gelatinise, and the Maillard reaction creates complex flavours and browning on the surface of meats and bread.

Balanced nutrition requires a mix of macronutrients — carbohydrates, proteins, and fats — along with vitamins and minerals. Carbohydrates are the body's primary energy source, broken down into glucose during digestion. Whole grains provide fibre that supports digestive health.

Proteins are chains of amino acids essential for building and repairing tissues. Complete proteins, found in animal products and soy, contain all nine essential amino acids. Combining legumes with grains, as many traditional cuisines do, also yields a complete amino acid profile.

Fats serve crucial roles in hormone production, cell membrane integrity, and absorption of fat-soluble vitamins A, D, E, and K. Unsaturated fats from olive oil, nuts, and avocados are associated with cardiovascular benefits, while excessive trans and saturated fats increase health risks.

Fermentation is one of the oldest food preservation techniques, used to produce yoghurt, cheese, bread, and beverages like beer and wine. Beneficial microorganisms convert sugars into acids or alcohol, extending shelf life while enhancing flavour and introducing probiotics that support gut health.

Space exploration entered a new era when NASA's Perseverance rover landed in Jezero Crater on Mars in February 2021. The rover carries instruments designed to search for signs of ancient microbial life and to collect rock samples for future return to Earth.

The Ingenuity helicopter, which travelled to Mars aboard Perseverance, demonstrated powered flight on another planet for the first time. Its successful flights proved that rotorcraft technology can operate in the thin Martian atmosphere, which is only about one percent as dense as Earth's.

SpaceX's Starship vehicle is being developed as a fully reusable transportation system capable of carrying humans to Mars. The stainless-steel rocket uses Raptor engines fuelled by liquid methane and liquid oxygen, propellants that could theoretically be manufactured on Mars using local resources.

International collaboration is key to future Mars missions. The European Space Agency, JAXA, and ISRO have all contributed orbiter and lander missions, while NASA's Artemis programme aims to establish a sustainable presence on the Moon as a stepping stone to Mars.
""".strip()

paragraphs = [p.strip() for p in CORPUS.split("\n\n") if p.strip()]
print(f"Paragraphs: {len(paragraphs)}")
for i, p in enumerate(paragraphs):
    print(f"  [{i:>2}] {p[:80]}...")

## 4 — Implementing from Scratch

### Step 1 — Split Text into Sentences

We need **sentence-level** granularity. A simple regex handles 90 % of English text.
For production you would use spaCy or NLTK, but regex keeps dependencies minimal.

In [ ]:
import re

def split_sentences(text):
    """Split text into sentences using regex."""
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in parts if len(s.strip()) > 10]

sentences = split_sentences(CORPUS)
print(f"Total sentences: {len(sentences)}")
for i, s in enumerate(sentences[:8]):
    print(f"  [{i:>2}] {s[:90]}...")
print(f"  ... ({len(sentences) - 8} more)")

### Step 2 — Embed Each Sentence

We encode every sentence into a 768-dimensional vector using `bge-base-en-v1.5`.
The `normalize_embeddings=True` flag makes every vector unit-length, so the dot product
equals cosine similarity — no extra normalisation needed later.

### Step 3 — Cosine Similarity Between Consecutive Sentences

Because the embeddings are L2-normalised, cosine similarity is simply the dot product:

$$\text{sim}_i = \mathbf{e}_i \cdot \mathbf{e}_{i+1}$$

We compute this for every consecutive pair, giving us an array of length *n − 1*.

In [ ]:
# Step 2: embed sentences
embeddings = embedder.encode(sentences, normalize_embeddings=True, show_progress_bar=True)
embeddings = np.array(embeddings)
print(f"Embedding matrix shape: {embeddings.shape}")
print(f"Norm of first vector:   {np.linalg.norm(embeddings[0]):.4f}  (should be ≈1.0)")

# Step 3: consecutive cosine similarities
similarities = np.array([
    np.dot(embeddings[i], embeddings[i + 1])
    for i in range(len(embeddings) - 1)
])
print(f"\nSimilarities shape: {similarities.shape}")
print(f"Mean: {similarities.mean():.4f}  Std: {similarities.std():.4f}")
print(f"Min:  {similarities.min():.4f}   Max: {similarities.max():.4f}")
print("\nFirst 10 consecutive similarities:")
for i, s in enumerate(similarities[:10]):
    print(f"  sim({i:>2}, {i+1:>2}) = {s:.4f}")

### Step 4 — Identify Breakpoints

A **breakpoint** is a position where the similarity is notably lower than its neighbours.
The simplest approach: pick a threshold *T* and mark every index where `sim_i < T`.
For a first pass we use the **25th percentile** of the similarity distribution.

### Step 5 — Group Sentences into Semantic Chunks

With breakpoints identified, we slice the sentence list at each breakpoint.
Each slice becomes one chunk.

In [ ]:
def find_breakpoints(similarities, threshold):
    """Return indices where similarity is below the threshold."""
    return [i + 1 for i, s in enumerate(similarities) if s < threshold]

def build_chunks(sentences, breakpoints):
    """Group sentences into chunks separated at breakpoint indices."""
    chunks, start = [], 0
    for bp in sorted(breakpoints):
        chunk_text = " ".join(sentences[start:bp])
        if chunk_text.strip():
            chunks.append(chunk_text)
        start = bp
    tail = " ".join(sentences[start:])
    if tail.strip():
        chunks.append(tail)
    return chunks

threshold_pct25 = float(np.percentile(similarities, 25))
breakpoints = find_breakpoints(similarities, threshold_pct25)
chunks_v1 = build_chunks(sentences, breakpoints)

print(f"Threshold (25th percentile): {threshold_pct25:.4f}")
print(f"Breakpoints: {len(breakpoints)}  at positions {breakpoints}")
print(f"Chunks: {len(chunks_v1)}\n")
for i, c in enumerate(chunks_v1):
    print(f"Chunk {i} ({len(c.split())} words): {c[:120]}...")

## 5 — Visualizing the Similarity Curve

The plot below shows cosine similarity between consecutive sentences.
Red dashed lines mark breakpoints; the green line is the threshold.
You should see dips at the transitions between our four topics.

In [ ]:
import matplotlib.pyplot as plt

def plot_similarities(sims, bps, threshold, title="Cosine Similarity"):
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(sims, marker="o", markersize=3, linewidth=1, label="Cosine sim")
    ax.axhline(y=threshold, color="green", linestyle="--", linewidth=1, label=f"Threshold = {threshold:.3f}")
    for bp in bps:
        ax.axvline(x=bp - 1, color="red", linestyle="--", alpha=0.6)
    ax.set_xlabel("Sentence index (i → i+1)")
    ax.set_ylabel("Cosine similarity")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_similarities(similarities, breakpoints, threshold_pct25,
                  title="Similarity Curve — Breakpoints at 25th Percentile")

## 6 — Threshold Selection Strategies

The threshold *T* directly controls chunk count:
- **Too high** → many tiny chunks (over-segmentation).
- **Too low** → few large chunks (under-segmentation).

### 6.1 — Percentile-Based
Pick *T* = the *p*-th percentile of the similarity distribution. Default: 10th–30th percentile.

### 6.2 — Absolute
Set *T* to a fixed value (e.g., 0.50). Simple but not adaptive to the distribution.

### 6.3 — Standard-Deviation Based
*T* = mean − *k* × std. Similarities more than *k* standard deviations below the mean are breakpoints.

In [ ]:
results = {}
mu, sigma = similarities.mean(), similarities.std()

print("Strategy              Threshold  Breakpoints  Chunks")
print("─" * 55)

for pct in [10, 25, 50]:
    t = float(np.percentile(similarities, pct))
    bps = find_breakpoints(similarities, t)
    ch  = build_chunks(sentences, bps)
    results[f"pct-{pct}"] = (t, bps, ch)
    print(f"Percentile {pct:>2}         {t:.4f}     {len(bps):>3}          {len(ch)}")

for abs_t in [0.40, 0.50, 0.60]:
    bps = find_breakpoints(similarities, abs_t)
    ch  = build_chunks(sentences, bps)
    results[f"abs-{abs_t}"] = (abs_t, bps, ch)
    print(f"Absolute {abs_t:.2f}         {abs_t:.4f}     {len(bps):>3}          {len(ch)}")

for k in [0.5, 1.0, 1.5]:
    t = float(mu - k * sigma)
    bps = find_breakpoints(similarities, t)
    ch  = build_chunks(sentences, bps)
    results[f"std-{k}"] = (t, bps, ch)
    print(f"Std-dev k={k:.1f}         {t:.4f}     {len(bps):>3}          {len(ch)}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, pct in zip(axes, [10, 25, 50]):
    t, bps, ch = results[f"pct-{pct}"]
    ax.plot(similarities, marker="o", markersize=2, linewidth=1)
    ax.axhline(y=t, color="green", linestyle="--", linewidth=1)
    for bp in bps:
        ax.axvline(x=bp - 1, color="red", linestyle="--", alpha=0.5)
    ax.set_title(f"{pct}th percentile: T={t:.3f} → {len(ch)} chunks")
    ax.set_ylabel("Cosine sim")

axes[-1].set_xlabel("Sentence pair index")
plt.suptitle("Effect of Percentile Threshold on Chunk Boundaries", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 7 — Alternative: Sliding-Window Smoothing

Sentence-level similarity can be noisy — one unusually short or ambiguous sentence can
create a false breakpoint. A **sliding window** averages similarity over a neighbourhood,
producing a smoother signal:

$$\text{smoothed}_i = \frac{1}{2w+1} \sum_{j=i-w}^{i+w} \text{sim}_j$$

where *w* is the half-window size. Edge values are handled by padding.

In [ ]:
def smooth_similarities(sims, window=3):
    """Apply a centred moving-average of width 2*window+1."""
    kernel = np.ones(2 * window + 1) / (2 * window + 1)
    padded = np.pad(sims, window, mode="edge")
    return np.convolve(padded, kernel, mode="valid")

smoothed = smooth_similarities(similarities, window=2)
threshold_smooth = float(np.percentile(smoothed, 25))
bps_smooth = find_breakpoints(smoothed, threshold_smooth)
chunks_smooth = build_chunks(sentences, bps_smooth)

print(f"Smoothed threshold (25th pct): {threshold_smooth:.4f}")
print(f"Breakpoints: {len(bps_smooth)}  →  Chunks: {len(chunks_smooth)}\n")

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(similarities, marker="o", markersize=2, linewidth=1)
axes[0].axhline(y=threshold_pct25, color="green", linestyle="--")
for bp in breakpoints:
    axes[0].axvline(x=bp-1, color="red", linestyle="--", alpha=0.5)
axes[0].set_title(f"Raw signal → {len(chunks_v1)} chunks")
axes[0].set_ylabel("Cosine sim")

axes[1].plot(smoothed, marker="o", markersize=2, linewidth=1, color="orange")
axes[1].axhline(y=threshold_smooth, color="green", linestyle="--")
for bp in bps_smooth:
    axes[1].axvline(x=bp-1, color="red", linestyle="--", alpha=0.5)
axes[1].set_title(f"Smoothed (window=2) → {len(chunks_smooth)} chunks")
axes[1].set_ylabel("Cosine sim")
axes[1].set_xlabel("Sentence pair index")
plt.tight_layout()
plt.show()

## 8 — Choosing Our Working Chunks

For the rest of the notebook we use the **25th-percentile** threshold on the
**raw** (unsmoothed) similarity signal — a solid default that balances granularity
and coherence.

In [ ]:
chosen_chunks = chunks_v1
print(f"Working with {len(chosen_chunks)} semantic chunks\n")
for i, c in enumerate(chosen_chunks):
    words = len(c.split())
    print(f"Chunk {i} — {words} words — {c[:100]}...\n")

## 9 — Complete RAG Pipeline

Now we wire everything into a retrieval-augmented generation pipeline:

1. **Embed** every chunk with `bge-base-en-v1.5`.
2. **Index** the embeddings in a FAISS flat-IP index (inner product = cosine for unit vectors).
3. **Retrieve** the top-*k* chunks for a user query.
4. **Generate** an answer with Qwen3-8B, grounding it in the retrieved context.

In [ ]:
import faiss

chunk_embeddings = embedder.encode(chosen_chunks, normalize_embeddings=True, show_progress_bar=True)
chunk_embeddings = np.array(chunk_embeddings).astype("float32")

dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)
print(f"FAISS index: {index.ntotal} vectors, dim={dim}")

In [ ]:
def retrieve(query, faiss_idx, chunks, k=3):
    """Embed the query, search FAISS, return top-k chunks with scores."""
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = faiss_idx.search(q_emb, k)
    return [{"chunk_id": int(idx), "score": float(sc), "text": chunks[idx]}
            for sc, idx in zip(scores[0], indices[0])]

def rag(question, faiss_idx, chunks, k=3, max_new_tokens=300):
    """Full RAG: retrieve context, build prompt, generate answer."""
    hits = retrieve(question, faiss_idx, chunks, k=k)
    context = "\n\n".join(f"[Chunk {h['chunk_id']}] {h['text']}" for h in hits)
    prompt = f"""Answer the question using ONLY the provided context.
If the context does not contain the answer, say "I don't know."

Context:
{context}

Question: {question}
Answer:"""
    answer = generate(prompt, max_new_tokens=max_new_tokens)
    return {"question": question, "answer": answer, "chunks_used": [h["chunk_id"] for h in hits]}

# Test
result = rag("What is the Calvin cycle and where does it occur?", index, chosen_chunks, k=2)
print(f"Q: {result['question']}")
print(f"Chunks: {result['chunks_used']}")
print(f"A: {result['answer']}")

### 9.1 — Testing Across Topics

Queries that span all four topics confirm the retriever picks the right chunks.

In [ ]:
test_questions = [
    "Who invented the World Wide Web and when?",
    "What role do fats play in human nutrition?",
    "What is the Ingenuity helicopter and what did it achieve on Mars?",
    "How do C4 plants reduce photorespiration?",
    "What is fermentation and what foods does it produce?",
]

for q in test_questions:
    r = rag(q, index, chosen_chunks, k=2, max_new_tokens=150)
    print(f"Q: {r['question']}")
    print(f"   Chunks: {r['chunks_used']}")
    print(f"   A: {r['answer'][:200]}")
    print()

## 10 — Comparison: Fixed-Size vs. Semantic Chunking

To quantify the benefit of semantic chunking, we build a second index using
**fixed-size chunks** (300 characters, 50-character overlap) and compare
retrieval relevance on the same queries.

### Why fixed-size chunks hurt retrieval

1. **Mid-sentence cuts** — a chunk ends in the middle of a thought.
2. **Topic mixing** — a single chunk may span two unrelated topics because the boundary fell between them.

Semantic chunks avoid both issues because boundaries are placed where meaning naturally shifts.

In [ ]:
def fixed_size_chunks(text, size=300, overlap=50):
    """Split text into fixed-size character windows."""
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        chunks.append(text[start:end])
        start += size - overlap
    return chunks

fix_chunks = fixed_size_chunks(CORPUS, size=300, overlap=50)
fix_embs = embedder.encode(fix_chunks, normalize_embeddings=True, show_progress_bar=True).astype("float32")
fix_index = faiss.IndexFlatIP(fix_embs.shape[1])
fix_index.add(fix_embs)

print(f"Fixed-size: {len(fix_chunks)} chunks  |  Semantic: {len(chosen_chunks)} chunks")
print(f"\nSample fixed chunk (last 80 chars):")
print(f"  ...{fix_chunks[3][-80:]}")
print(f"  Ends mid-sentence? {not fix_chunks[3].rstrip().endswith('.')}")

In [ ]:
comparison_queries = [
    "How does photosynthesis work in C4 plants?",
    "What technologies power the modern Internet?",
    "What are the health benefits of fermentation?",
]

for q in comparison_queries:
    sem_hits = retrieve(q, index, chosen_chunks, k=2)
    fix_hits = retrieve(q, fix_index, fix_chunks, k=2)
    print(f"Query: {q}")
    print(f"  SEMANTIC scores: {[round(h['score'], 3) for h in sem_hits]}")
    print(f"  FIXED    scores: {[round(h['score'], 3) for h in fix_hits]}")
    top_fix = fix_hits[0]["text"]
    print(f"  Top fixed chunk (last 60 chars): ...{top_fix[-60:]}")
    print(f"  Cut mid-sentence? {not top_fix.rstrip().endswith('.')}")
    print()

### 10.1 — LLM Answer Quality Comparison

Same question to both pipelines — observe the difference in answer quality.

In [ ]:
eval_q = "Explain how the Maillard reaction works and its role in cooking."

sem_result = rag(eval_q, index, chosen_chunks, k=2, max_new_tokens=200)
print("SEMANTIC RAG answer:")
print(f"  Chunks: {sem_result['chunks_used']}")
print(f"  {sem_result['answer'][:300]}\n")

fix_hits = retrieve(eval_q, fix_index, fix_chunks, k=2)
fix_ctx = "\n\n".join(f"[Chunk {h['chunk_id']}] {h['text']}" for h in fix_hits)
fix_prompt = f"""Answer the question using ONLY the provided context.
If the context does not contain the answer, say "I don't know."

Context:
{fix_ctx}

Question: {eval_q}
Answer:"""
fix_answer = generate(fix_prompt, max_new_tokens=200)
print("FIXED-SIZE RAG answer:")
print(f"  Chunks: {[h['chunk_id'] for h in fix_hits]}")
print(f"  {fix_answer[:300]}")

In [ ]:
print("=== Looking for cross-topic fixed chunks ===\n")
for i, c in enumerate(fix_chunks):
    has_plant   = any(w in c.lower() for w in ["photosynthesis", "chloroplast", "transpiration"])
    has_net     = any(w in c.lower() for w in ["internet", "arpanet", "tcp"])
    has_cook    = any(w in c.lower() for w in ["cooking", "nutrition", "maillard"])
    has_space   = any(w in c.lower() for w in ["mars", "rover", "spacex"])
    topics = sum([has_plant, has_net, has_cook, has_space])
    if topics >= 2:
        print(f"Fixed chunk {i} spans MULTIPLE topics:")
        print(f"  {c[:200]}...\n")
        break
else:
    # Show a boundary-cut chunk as fallback
    mid = len(fix_chunks) // 3
    print(f"Fixed chunk {mid} boundary cut:")
    print(f"  ...{fix_chunks[mid][-100:]}\n")

print("=== Semantic chunks stay within one topic ===\n")
for i, c in enumerate(chosen_chunks[:3]):
    print(f"Semantic chunk {i}: {c[:120]}...")

## 11 — Advanced: Hierarchical Semantic Chunking

For very long documents we can apply semantic chunking **recursively**:

1. **Level 1 — Paragraph chunking:** Split by paragraphs, embed each paragraph,
   find breakpoints → produces *sections*.
2. **Level 2 — Sentence chunking:** Within each section, apply sentence-level
   semantic chunking → produces fine-grained chunks.

This two-level approach captures both **macro** structure (topic shifts between sections)
and **micro** structure (sub-topic shifts within a section).

In [ ]:
def hierarchical_semantic_chunk(text, embed_fn, coarse_pct=25, fine_pct=30):
    """Two-level semantic chunking: paragraph-level then sentence-level."""
    # Level 1: paragraph-level
    paras = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 20]
    if len(paras) < 2:
        return [text]

    para_embs = embed_fn(paras, normalize_embeddings=True)
    para_sims = np.array([np.dot(para_embs[i], para_embs[i+1]) for i in range(len(para_embs)-1)])
    para_thresh = float(np.percentile(para_sims, coarse_pct))
    para_bps = find_breakpoints(para_sims, para_thresh)
    sections = build_chunks(paras, para_bps)

    print(f"Level 1 (paragraph): {len(paras)} paragraphs → {len(sections)} sections  (T={para_thresh:.3f})")

    # Level 2: sentence-level within each section
    final_chunks = []
    for sec_i, section in enumerate(sections):
        sents = split_sentences(section)
        if len(sents) < 3:
            final_chunks.append(section)
            continue
        sent_embs = embed_fn(sents, normalize_embeddings=True)
        sent_sims = np.array([np.dot(sent_embs[i], sent_embs[i+1]) for i in range(len(sent_embs)-1)])
        sent_thresh = float(np.percentile(sent_sims, fine_pct))
        sent_bps = find_breakpoints(sent_sims, sent_thresh)
        sub_chunks = build_chunks(sents, sent_bps)
        print(f"  Section {sec_i}: {len(sents)} sentences → {len(sub_chunks)} sub-chunks")
        final_chunks.extend(sub_chunks)

    return final_chunks

hier_chunks = hierarchical_semantic_chunk(CORPUS, embedder.encode)
print(f"\nTotal hierarchical chunks: {len(hier_chunks)}")

In [ ]:
print(f"Hierarchical chunks: {len(hier_chunks)}\n")
for i, c in enumerate(hier_chunks):
    print(f"H-Chunk {i:>2} ({len(c.split()):>3} words): {c[:100]}...")

print(f"\n{'='*50}")
print(f"Flat semantic (25th pct):  {len(chosen_chunks)} chunks")
print(f"Hierarchical (25/30 pct):  {len(hier_chunks)} chunks")
print(f"Fixed-size (300/50):       {len(fix_chunks)} chunks")

### 11.1 — RAG with Hierarchical Chunks

In [ ]:
hier_embs = embedder.encode(hier_chunks, normalize_embeddings=True, show_progress_bar=True).astype("float32")
hier_index = faiss.IndexFlatIP(hier_embs.shape[1])
hier_index.add(hier_embs)

test_q = "What fuel does SpaceX's Starship use?"
for label, idx, chs in [("Flat semantic", index, chosen_chunks),
                          ("Hierarchical", hier_index, hier_chunks),
                          ("Fixed-size", fix_index, fix_chunks)]:
    hits = retrieve(test_q, idx, chs, k=2)
    print(f"{label:>17}: scores={[round(h['score'], 3) for h in hits]}")
    print(f"{'':>17}  top = {hits[0]['text'][:100]}...")
    print()

## 12 — Chunk Size Distributions

Semantic methods produce variable-length chunks that follow the document's natural
structure. Fixed-size is uniform by construction but semantically arbitrary.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, (label, chs) in zip(axes, [
    ("Flat Semantic", chosen_chunks),
    ("Hierarchical", hier_chunks),
    ("Fixed-Size", fix_chunks),
]):
    lengths = [len(c.split()) for c in chs]
    ax.hist(lengths, bins=15, edgecolor="black", alpha=0.7)
    ax.set_title(f"{label} ({len(chs)} chunks)")
    ax.set_xlabel("Words per chunk")
    ax.axvline(np.mean(lengths), color="red", linestyle="--", label=f"mean={np.mean(lengths):.0f}")
    ax.legend()

axes[0].set_ylabel("Count")
plt.suptitle("Chunk Size Distributions", fontsize=13)
plt.tight_layout()
plt.show()

## 13 — Reusable `SemanticChunker` Class

A clean, copy-paste-ready class that encapsulates the full algorithm with
configurable threshold strategy and optional sliding-window smoothing.

In [ ]:
class SemanticChunker:
    """Semantic chunking with configurable threshold strategy."""

    def __init__(self, embed_fn, method="percentile", value=25, window=0):
        self.embed_fn = embed_fn
        self.method   = method   # "percentile", "absolute", "stddev"
        self.value    = value    # percentile number, absolute float, or k for stddev
        self.window   = window   # smoothing half-window (0 = no smoothing)

    def chunk(self, text):
        sents = split_sentences(text)
        if len(sents) < 2:
            return [text]
        embs = self.embed_fn(sents, normalize_embeddings=True)
        sims = np.array([np.dot(embs[i], embs[i+1]) for i in range(len(embs)-1)])
        if self.window > 0:
            sims = smooth_similarities(sims, window=self.window)
        if self.method == "percentile":
            threshold = float(np.percentile(sims, self.value))
        elif self.method == "absolute":
            threshold = self.value
        elif self.method == "stddev":
            threshold = float(sims.mean() - self.value * sims.std())
        else:
            raise ValueError(f"Unknown method: {self.method}")
        bps = find_breakpoints(sims, threshold)
        return build_chunks(sents, bps)

# Demo: default percentile
chunker = SemanticChunker(embedder.encode, method="percentile", value=25)
demo = chunker.chunk(CORPUS)
print(f"SemanticChunker (percentile-25): {len(demo)} chunks")
for i, c in enumerate(demo):
    print(f"  Chunk {i}: {len(c.split())} words — {c[:80]}...")

## 14 — End-to-End: One Function for Everything

A single function that takes raw text and a query and returns a grounded answer.

In [ ]:
def semantic_rag(text, question, k=3, method="percentile", value=25, max_new_tokens=300):
    """End-to-end semantic chunking RAG in one call."""
    chunker = SemanticChunker(embedder.encode, method=method, value=value)
    chunks = chunker.chunk(text)
    print(f"Chunked into {len(chunks)} pieces")

    embs = embedder.encode(chunks, normalize_embeddings=True).astype("float32")
    idx = faiss.IndexFlatIP(embs.shape[1])
    idx.add(embs)

    q_emb = embedder.encode([question], normalize_embeddings=True).astype("float32")
    scores, ids = idx.search(q_emb, k)
    context = "\n\n".join(f"[Chunk {int(j)}] {chunks[j]}" for j in ids[0])
    print(f"Retrieved chunks: {ids[0].tolist()} scores={[round(s, 3) for s in scores[0]]}")

    prompt = f"""Answer the question using ONLY the provided context.
If the context does not contain the answer, say "I don't know."

Context:
{context}

Question: {question}
Answer:"""
    return generate(prompt, max_new_tokens=max_new_tokens)

answer = semantic_rag(CORPUS, "What is TCP/IP and when was it adopted?")
print(f"\nAnswer: {answer}")

In [ ]:
for q in [
    "How do guard cells regulate transpiration?",
    "What is the difference between complete and incomplete proteins?",
    "What instruments does the Perseverance rover carry?",
]:
    print(f"Q: {q}")
    a = semantic_rag(CORPUS, q, k=2, max_new_tokens=150)
    print(f"A: {a}\n{'─'*80}\n")

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** swap out the retriever/chunker and measure the impact on recall. Document what changes and why.

**Exercise 2 — Build:** add a new document type and test retrieval quality. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** implement a simple version of the algorithm from scratch.

## 15 — Summary & Key Takeaways

| Aspect | Fixed-Size Chunking | Semantic Chunking |
|---|---|---|
| **Boundary logic** | Every *N* characters | Where meaning shifts |
| **Coherence** | May cut mid-sentence / topic | Each chunk is a semantic unit |
| **Implementation** | Trivial | Requires embeddings + threshold |
| **Retrieval quality** | Acceptable for simple docs | Superior for multi-topic docs |
| **Computational cost** | O(1) per chunk | O(n) embeddings + O(n) similarity |

**Practical guidelines:**
- Use the **percentile** method (10th–30th) as a starting default.
- Add **sliding-window smoothing** (window=2–3) if the text is noisy or has short sentences.
- For long documents, try **hierarchical** chunking — paragraph-level first, then sentence-level.
- Always **normalize** embeddings so dot product = cosine similarity; it simplifies everything.

**What we built:**
- A from-scratch semantic chunker in pure Python + NumPy.
- Three threshold strategies (percentile, absolute, std-dev) with visual comparison.
- A sliding-window smoothing variant.
- A reusable `SemanticChunker` class.
- A complete RAG pipeline with FAISS retrieval and Qwen3-8B generation.
- Hierarchical (two-level) semantic chunking.
- Head-to-head comparison proving semantic > fixed-size on multi-topic text.

## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the rag/ directory

---

## 🧭 Navigation

⬅️ **Previous:** [Self Rag](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/self_rag.ipynb) | ➡️ **Next:** [Simple Csv Rag](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/simple_csv_rag.ipynb)